# ARGUS Phase 3 — Classifier Evaluation & Threshold Calibration

Evaluates the fine-tuned classifier on **same-day normal controls** (not just val split).
Produces ROC-AUC, PR-AUC, threshold sweep, confusion matrices, and real alert-engine outputs.

**Prerequisites:** Run the Phase 2.5 + Phase 3 fine-tune notebooks first.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, time

# ── Paths ──
DATA_ROOT   = Path("/kaggle/input/datasets/nightingale21/argus-tokenized-58day-verified/data")
VOCAB_PATH  = DATA_ROOT / "vocab.json"
SESSIONS_DIR = DATA_ROOT / "sessions"           # day_XX.parquet files
VAL_MANIFEST = DATA_ROOT / "tokenized" / "sessions_val.pt"

REDTEAM_PATH = Path("/kaggle/input/datasets/nightingale21/attacker/redteam.txt")

REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = Path("/kaggle/working/argus-log-intelligence-platform")
REFRESH_REPO = True

# Phase 2.5 outputs
ATTACK_MANIFEST = Path("/kaggle/working/attack_sessions/attack_sessions.pt")

# Phase 3 fine-tune outputs
CLASSIFIER_CKPT = Path("/kaggle/working/argus_finetuned/best_classifier.pt")

# Phase 3 eval outputs
EVAL_OUT = Path("/kaggle/working/argus_phase3_eval")
EVAL_OUT.mkdir(parents=True, exist_ok=True)

# Attack days (from Phase 2.5 scan)
ATTACK_DAYS = [
    2, 3, 6, 7, 8, 9, 10, 13, 14, 15, 16,
    21, 22, 23, 27, 28, 29, 30
]


In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

def run_stream(command, cwd=None, env=None):
    print("$", " ".join(str(p) for p in command), flush=True)
    proc = subprocess.Popen(
        [str(p) for p in command], cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc})")

if REFRESH_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run_stream(["git", "clone", REPO_URL, str(REPO_DIR)])

run_stream([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.37.0", "pyarrow>=14.0.0", "tqdm>=4.67.1",
    "pyyaml>=6.0"])

sys.path.insert(0, str(REPO_DIR))
print("Repo ready:", REPO_DIR)


## Step 1: Verify prerequisites

In [ ]:
required = {
    "Classifier": CLASSIFIER_CKPT,
    "Attack manifest": ATTACK_MANIFEST,
    "Val manifest": VAL_MANIFEST,
    "Vocab": VOCAB_PATH,
    "Redteam": REDTEAM_PATH,
}
for name, p in required.items():
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}: {name} → {p}")
    if not p.exists():
        raise FileNotFoundError(f"{name} not found: {p}")

# Load classifier info
ckpt = torch.load(CLASSIFIER_CKPT, map_location="cpu", weights_only=False)
print(f"\nClassifier: epoch {ckpt.get('epoch')}, train F1={ckpt.get('best_f1', 0):.4f}")
print(f"Val metrics: {ckpt.get('val_metrics', {})}")


## Step 2: Build same-day normal controls

Attack sessions come from days 02–30. For a fair evaluation, our normal controls
should also come from those same days (not from the temporal val split days 41+).

We load session parquets for attack days, remove attack sessions, tokenize a sample
of normal sessions, and save as a manifest.

In [ ]:
import pandas as pd
import numpy as np
from src.parsing.log_tokenizer import LogTokenizer

CONTROL_DIR = EVAL_OUT / "normal_controls"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_MANIFEST = CONTROL_DIR / "normal_controls.pt"

# Load redteam data to identify attack sessions
redteam_lines = REDTEAM_PATH.read_text().strip().splitlines()
attack_users = set()
for line in redteam_lines:
    parts = line.strip().split(",")
    if len(parts) >= 3:
        attack_users.add(parts[1].strip())  # src_user
print(f"Red team users: {len(attack_users)}")

# Load tokenizer
tokenizer = LogTokenizer(str(VOCAB_PATH))
print(f"Vocab: {len(tokenizer.vocab)} tokens, max_len={tokenizer.max_len}")

# Sample normal sessions from attack days
MAX_NORMAL_PER_DAY = 1000
all_normal_ids = []
all_normal_masks = []
total_scanned = 0
total_skipped_attack = 0

for day_num in ATTACK_DAYS:
    parquet_path = SESSIONS_DIR / f"day_{day_num:02d}.parquet"
    if not parquet_path.exists():
        print(f"  day_{day_num:02d}: not found, skipping")
        continue

    df = pd.read_parquet(parquet_path)
    total_scanned += len(df)

    # Filter out sessions from attack users
    if "user" in df.columns:
        user_col = "user"
    elif "src_user" in df.columns:
        user_col = "src_user"
    elif "user_id" in df.columns:
        user_col = "user_id"
    else:
        user_col = None

    if user_col:
        attack_mask = df[user_col].isin(attack_users)
        total_skipped_attack += attack_mask.sum()
        normal_df = df[~attack_mask]
    else:
        normal_df = df

    # Sample up to MAX_NORMAL_PER_DAY
    if len(normal_df) > MAX_NORMAL_PER_DAY:
        normal_df = normal_df.sample(n=MAX_NORMAL_PER_DAY, random_state=42)

    # Tokenize each session
    day_count = 0
    events_col = None
    for col in ("events", "event_tokens", "token_sequence"):
        if col in normal_df.columns:
            events_col = col
            break

    for _, row in normal_df.iterrows():
        try:
            if events_col and isinstance(row[events_col], (list, np.ndarray)):
                events = row[events_col]
                if isinstance(events[0], dict):
                    result = tokenizer.tokenize_session({"events": events})
                else:
                    # Already tokenized as int list
                    seq = list(events)
                    if len(seq) > tokenizer.max_len:
                        seq = seq[:tokenizer.max_len]
                    ids = torch.tensor(seq, dtype=torch.long)
                    mask = (ids != tokenizer.pad_token).long()
                    all_normal_ids.append(ids)
                    all_normal_masks.append(mask)
                    day_count += 1
                    continue
            else:
                # Try to build from row columns
                result = tokenizer.tokenize_session(row.to_dict())

            if isinstance(result, dict):
                ids = result.get("input_ids", result.get("token_ids"))
                if isinstance(ids, torch.Tensor):
                    all_normal_ids.append(ids)
                else:
                    ids_t = torch.tensor(ids, dtype=torch.long)
                    all_normal_ids.append(ids_t)
                mask = result.get("attention_mask")
                if mask is None:
                    mask = (all_normal_ids[-1] != tokenizer.pad_token).long()
                elif not isinstance(mask, torch.Tensor):
                    mask = torch.tensor(mask, dtype=torch.long)
                all_normal_masks.append(mask)
                day_count += 1
        except Exception:
            continue

    print(f"  day_{day_num:02d}: {day_count} normal sessions tokenized")

print(f"\nTotal scanned: {total_scanned:,}")
print(f"Attack-user sessions skipped: {total_skipped_attack:,}")
print(f"Normal controls tokenized: {len(all_normal_ids):,}")


In [ ]:
# Save as manifest format for the evaluation script
if len(all_normal_ids) > 0:
    # Pad all to same length
    max_len = tokenizer.max_len
    padded_ids = []
    padded_masks = []
    for ids, mask in zip(all_normal_ids, all_normal_masks):
        if len(ids) < max_len:
            pad_len = max_len - len(ids)
            ids = torch.cat([ids, torch.full((pad_len,), tokenizer.pad_token, dtype=torch.long)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
        elif len(ids) > max_len:
            ids = ids[:max_len]
            mask = mask[:max_len]
        padded_ids.append(ids)
        padded_masks.append(mask)

    ids_tensor = torch.stack(padded_ids)
    masks_tensor = torch.stack(padded_masks)

    # Save as single chunk
    chunk_dir = CONTROL_DIR / "normal_controls_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = chunk_dir / "chunk_0000.pt"
    torch.save({"input_ids": ids_tensor, "attention_mask": masks_tensor}, chunk_path)

    manifest = {
        "format": "tokenized_session_chunk_manifest_v1",
        "chunks": [str(chunk_path.relative_to(CONTROL_DIR))],
        "total_sessions": len(padded_ids),
    }
    torch.save(manifest, CONTROL_MANIFEST)
    print(f"Normal controls manifest: {CONTROL_MANIFEST}")
    print(f"  {len(padded_ids):,} sessions, shape {ids_tensor.shape}")
else:
    print("No normal controls built — falling back to val manifest")
    CONTROL_MANIFEST = VAL_MANIFEST


## Step 3: Run classifier evaluation

In [ ]:
# Determine which normal manifest to use
normal_manifest = str(CONTROL_MANIFEST) if CONTROL_MANIFEST.exists() else str(VAL_MANIFEST)

eval_cmd = [
    sys.executable, "-m", "scripts.evaluate_attack_classifier",
    "--classifier", str(CLASSIFIER_CKPT),
    "--attack-manifest", str(ATTACK_MANIFEST),
    "--normal-manifest", normal_manifest,
    "--out", str(EVAL_OUT),
    "--max-normal", "10000",
    "--batch-size", "128",
    "--num-workers", "2",
]
run_stream(eval_cmd, cwd=REPO_DIR,
           env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Step 4: Review results

In [ ]:
report_path = EVAL_OUT / "evaluation_report.json"
report = json.loads(report_path.read_text())

print("=" * 60)
print("ARGUS Phase 3 — Classifier Evaluation Report")
print("=" * 60)
print(f"\nAttack sessions:  {report['n_attack']}")
print(f"Normal controls:  {report['n_normal']}")
print(f"\nROC-AUC:  {report['roc_auc']:.6f}")
print(f"PR-AUC:   {report['pr_auc']:.6f}")

print(f"\nAttack P(attack): mean={report['attack_prob_stats']['mean']:.4f} "
      f"std={report['attack_prob_stats']['std']:.4f}")
print(f"Normal P(attack): mean={report['normal_prob_stats']['mean']:.4f} "
      f"std={report['normal_prob_stats']['std']:.4f}")

print(f"\nBest threshold: {report['best_threshold']['threshold']:.2f}")
print(f"  F1={report['best_threshold']['f1']:.4f} "
      f"Prec={report['best_threshold']['precision']:.4f} "
      f"Rec={report['best_threshold']['recall']:.4f}")

print(f"\n── Threshold Sweep ──")
print(f"{'Thresh':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'FPR':>8}")
for row in report['threshold_sweep']:
    print(f"{row['threshold']:>8.2f} {row['precision']:>8.4f} {row['recall']:>8.4f} "
          f"{row['f1']:>8.4f} {row['fpr']:>8.4f}")

print(f"\n── Alert Engine (real outputs) ──")
ae = report['alert_engine_results']
print(f"  Alerts:    {ae['total_alerts']}")
print(f"  True pos:  {ae['true_attack_alerts']}")
print(f"  False pos: {ae['false_alerts']}")
print(f"  Precision: {ae['alert_precision']:.4f}")
print(f"  Severity:  {ae['severity_distribution']}")


## Step 5: Archive

In [ ]:
ARCHIVE = Path("/kaggle/working/argus_phase3_eval_archive")
ARCHIVE.mkdir(exist_ok=True)

for f in [
    EVAL_OUT / "evaluation_report.json",
    EVAL_OUT / "classifier_scores.csv",
    EVAL_OUT / "calibrated_thresholds.json",
    CLASSIFIER_CKPT,
    Path("/kaggle/working/argus_finetuned/finetune_history.json"),
]:
    if f.exists():
        shutil.copy2(f, ARCHIVE / f.name)
        print(f"  Archived: {f.name}")

archive = shutil.make_archive(str(ARCHIVE), "zip", root_dir=ARCHIVE)
print(f"\nArchive: {archive}")
print("Phase 3 evaluation complete!")
